In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGSMITH_ENDPOINT"]

## Sequential chain

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate, 
                                    ChatPromptTemplate)

base_url = "http://localhost:11434/"
model_name = "llama3.2:1b"

llm = ChatOllama(
    base_url = base_url,
    model = model_name,
    temperature = 0.8,
    num_predict = 256
)

system_prompt = SystemMessagePromptTemplate.from_template("You are a {school} teacher. You answer in short sentences.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {points} points.")
template = ChatPromptTemplate([system_prompt, human_prompt])
messages = template.invoke({"points": 3, "topic": "Mars", "school": "elementary school"})
response = llm.invoke(messages)
print(response.content)

In [ ]:
chain = template | llm
chain

In [ ]:
response = chain.invoke({"points": 3, "topic": "Mars", "school": "elementary school"})
print(response.content)

In [ ]:
response

### With stroutputparser
To get the content from the response without other metadata.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = template | llm | StrOutputParser()

response = chain.invoke({"points": 3, "topic": "Mars", "school": "elementary school"})
print(response)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate, 
                                    ChatPromptTemplate)
from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434/"
model_name = "llama3.2:1b"

llm = ChatOllama(
    base_url = base_url,
    model = model_name
)

system_prompt = SystemMessagePromptTemplate.from_template("You are a {school} teacher. You answer in short sentences.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {points} points.")
template = ChatPromptTemplate([system_prompt, human_prompt])

chain = template | llm | StrOutputParser()

response = chain.invoke({"points": 3, "topic": "Mars", "school": "elementary school"})
print(response)

### Multiple chains

In [ ]:
analyze_prompt = ChatPromptTemplate.from_template("""
Analyze this text: 
{response}

You need to tell me how difficult it is to understand.
Answer me in one sentence only.
""")

In [ ]:
chain2 = analyze_prompt | llm | StrOutputParser()
response = chain2.invoke({"response": response})
print(response)

In [ ]:
composed_chain = {"response": chain} | chain2
response = composed_chain.invoke({"points": 3, "topic": "Mars", "school": "elementary school"})
print(response)

## Parallel chains

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate, 
                                    ChatPromptTemplate)
from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434/"
model_name = "llama3.2:1b"

llm = ChatOllama(
    base_url = base_url,
    model = model_name
)

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template("You are a helpful assistant. You answer in short sentences.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me facts about {topic} in {points} points.")
template = ChatPromptTemplate([system_prompt, human_prompt])

fact_chain = template | llm | StrOutputParser()

response = fact_chain.invoke({"points": 3, "topic": "solar system"})
print(response)

In [ ]:
human_prompt = HumanMessagePromptTemplate.from_template("Write me a poem about {topic} in {lines} lines.")
template = ChatPromptTemplate([system_prompt, human_prompt])

poem_chain = template | llm | StrOutputParser()

response = poem_chain.invoke({"lines": 2, "topic": "solar system"})
print(response)

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(fact=fact_chain,
                                  poem=poem_chain)
parallel_chain

In [ ]:
response = parallel_chain.invoke({
  "points": 3,
  "lines": 2, 
  "topic": "solar system"
})

print("Fact chain output:")
print(response['fact'])
print()
print("Poem chain output:")
print(response['poem'])


## Chain Router
The router chain is used to route the output of a previous runnable to the next runnable based on the output of the previous runnable.

In [ ]:
prompt = """
Given the user review below, classify it as either being about "Positive" or "Negative".
Do not respond with more than one word.

Review: {review}
Classification:
"""

template = ChatPromptTemplate.from_template(prompt)

chain = template | llm | StrOutputParser()
review = "I am not happy with the service. It is not good."
chain.invoke({'review': review})

In [ ]:
positive_prompt = """
You are expert in writing reply for positive reviews.
You need to encourage the user to share their experience on social media.
Review: {review}
Answer:
"""

positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain = positive_template | llm | StrOutputParser()

In [ ]:
negative_prompt = """
You are expert in writing reply for negative reviews.
You need first to apologize for the inconvenience caused to the user.
You need to encourage the user to share their concern on following email:'farrel@gmail.com'.
Review: {review}
Answer:
"""

negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | StrOutputParser()

In [ ]:
def route(info):
  sentiment = info["sentiment"].lower()
  if "positive" in sentiment:
    return positive_chain
  else:
    return negative_chain

In [ ]:
route({"sentiment": "Positive"})

In [ ]:
from langchain_core.runnables import RunnableLambda

full_chain = {"sentiment": chain, "review": lambda x: x["review"]} | RunnableLambda(route)

In [ ]:
# review = "Thank you so much for providing such a great plateform for learning. I am really happy with the service."
review = "I am not happy with the service. It is not good."

response = full_chain.invoke({"review": review})
print(response)

## Custom runnable chain with RunnablePassthrough and RunnableLambda

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
def char_count(text):
  return len(text)

def word_count(text):
  return len(text.split())

In [ ]:
prompt = ChatPromptTemplate.from_template("Explain these inputs: {input1} and {input2} in three sentences.")

In [ ]:
chain = prompt | llm | StrOutputParser()
response = chain.invoke({"input1": "Earth", 
                         "input2": "Mars"})
print(response)

In [ ]:
chain = prompt | llm | StrOutputParser() | {"char_count": RunnableLambda(char_count),
                                            "word_count": RunnableLambda(word_count),
                                            "output": RunnablePassthrough()}

response = chain.invoke({"input1": "Earth", 
                         "input2": "Mars"})
print(response)

In [ ]:
print("Output:")
print(response['output'])
print()
print(f"Char count: {response['char_count']}")
print(f"Word count: {response['word_count']}")

## Custom chain with @chain decorator

In [ ]:
from langchain_core.runnables import chain

In [ ]:
@chain
def custom_chain(params):
  return {
    "fact": fact_chain.invoke(params),
    "poem": poem_chain.invoke(params)
  }

In [ ]:
params = {
  "points": 3,
  "lines": 2, 
  "topic": "solar system"
}

response = custom_chain.invoke(params)

print("Fact chain output:")
print(response['fact'])
print()
print("Poem chain output:")
print(response['poem'])
